# Preprocessing Spectra Data with presursor
- combine annotated Spectra from study with Spectra not annotaded

In [ ]:
import pandas as pd
from pyteomics import mgf
import re
import numpy as np
import json
import pickle    
from scipy.spatial.distance import pdist, squareform


In [ ]:
spectra = list(mgf.read("../../data/medical/RFA MSMS.mgf"))
print(f"Number of spectra: {len(spectra)}")

In [ ]:
df_spectra =pd.DataFrame(spectra)
print(df_spectra.columns)
df_spectra.head()

In [ ]:
DEFAULT_INSTRUMENT = "Orbitrap"
DEFAULT_COLLISION_ENERGY = 50.0
DEFAULT_SIMULATION_CHALLENGE = False

def retention_time(string):
    rt = re.search(r'\(([\d\.]+)_', string)
    return float(rt.group(1)) if rt else None

rows = []
big = 0
# create a json file
for inex, spec in df_spectra.iterrows():
    ID = spec["params"]["title"]
    new_ID = ID.replace("Unknown (", "")
    new_ID = new_ID.replace(")", "")
    rt = retention_time(ID)
    pepmass = spec["params"]["pepmass"][0]
    #print(f"ID {ID} has pepmass {pepmass}")
    if pepmass > 1000:
        big += 1
        continue
    peaks_json = np.column_stack((spec['m/z array'], spec['intensity array'])).tolist()

    entry = {
            "identifier": new_ID,
            "retention_time": rt,
            "precursor_formula": None,  # not available
            "precursor_mz": pepmass,
            "peaks_mz": [],
            "instrument_type": DEFAULT_INSTRUMENT,
            "collision_energy": DEFAULT_COLLISION_ENERGY,
            "simulation_challenge": DEFAULT_SIMULATION_CHALLENGE,
            "peaks_json": peaks_json
        }
    rows.append(entry)
json.dump(rows, open("../../data/medical/RFA MSMS_mz_precursor.json", "w"), indent=4)
print(f"Number of spectra with precursor mass above 1000: {big}")
print(f"Number of spectra with precursor mass below 1000: {len(rows)}")

## 3. Combine Datasets into one
- combine m/z (sirius annotated) and n (study annotated) into one spectra dataset

In [ ]:
# Build combined dataset of m/z and n spectra
mz_dataset = json.load(open("../../data/medical/RFA MSMS_mz_precursor.json"))
n_dataset = json.load(open("../../data/medical/RFA MSMS_n.json"))
msms_dataset = mz_dataset + n_dataset
print(f"Total number of MS/MS spectra: {len(msms_dataset)}")
occured = set()
nonunique = set()
count = 0
for spectrum in msms_dataset:
    spectrum_id = spectrum['identifier']
    if spectrum_id in occured:
        #print(f"Feature {spectrum_id} has a matching MS/MS spectrum")
        count += 1
        nonunique.add(spectrum_id)
    occured.add(spectrum_id)
#print(f"Number of features that occur multiple times:: {len(occured)}")
print(f"Total number of features non unique: {count}")
#print(f"First 10  feature IDs: {list(occured)[:10]}")
#print(f"First 10 non unique feature IDs: {list(nonunique)[:10]}")

In [ ]:
#Assign spectra with similar IDs an identifier based on occurences
identifier_counts = {}
new_msms_dataset = []
for spectrum in msms_dataset:
    spectrum_id = spectrum['identifier']
    if spectrum_id not in identifier_counts:
        identifier_counts[spectrum_id] = 1
    else:
        identifier_counts[spectrum_id] += 1
    new_id = f"{spectrum_id}_{identifier_counts[spectrum_id]}"
    new_spectrum = spectrum.copy()
    new_spectrum['identifier'] = new_id
    new_msms_dataset.append(new_spectrum)    

json.dump(new_msms_dataset, open("../../data/medical/RFA_MSMS_full_precursor.json", "w"), indent=4)